# Requirement 18: Separate untrusted-code

> Isolate AI-generated Python from the orchestrator, its credentials, and other sessions.

**Covered by:** Sandbox

**How it is fulfilled:** The generated code runs inside the sandbox micro virtual machine. The orchestrator's credentials live in the orchestrator process, never in the sandbox.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from helpers import *
sandbox, session_id = new_session()
sandbox.write_file("/tmp/untrusted.py", "import os, glob\nprint('env keys:', sorted(k for k in os.environ if 'SECRET' in k or 'CLIENT' in k or 'AZURE' in k))\nprint('azure cli config:', glob.glob('/root/.azure*') + glob.glob('/home/*/.azure*'))")
run = sandbox.exec("python /tmp/untrusted.py")
print(run.stdout)
check("no orchestrator credentials in the sandbox environment", "[]" in run.stdout.splitlines()[0])
check("no Azure CLI login state in the sandbox", run.stdout.strip().endswith("[]"))

env keys: []
azure cli config: []

PASS  no orchestrator credentials in the sandbox environment
PASS  no Azure CLI login state in the sandbox


In [3]:
secret = os.environ["ORCHESTRATOR_CLIENT_SECRET"]
check("the orchestrator's client secret is not on the sandbox disk",
      sandbox.exec(f"grep -rl --exclude-dir=proc --exclude-dir=sys -F '{secret}' /tmp /home /root /etc 2>/dev/null | wc -l").stdout.strip() == "0")
print("hostname inside the sandbox:", sandbox.exec("hostname").stdout.strip())
cleanup(sandbox)

PASS  the orchestrator's client secret is not on the sandbox disk
hostname inside the sandbox: adc-sandbox


**Caveat:** Separation from the orchestrator is by process and machine boundary. Escape from the microVM is a platform property that this demo cannot test.